## Camada Silver
<p>
Nesta camada, é realizada a leitura da camada bronze e é realizado o pré-processamento do dado. Por pré-processamento, entende-se:
<br>
<ul>
<li>Conversão de tipos</li>
<li>Padronização de textos</li>
<li>*Tratamento de dados inválidos</li>
</ul>
</p>

In [0]:
# Imports
from delta.tables import DeltaTable
from pyspark.sql import Column, DataFrame
from pyspark.sql import functions as F

In [0]:
CATALOGO     = "ANP_Combustiveis"
bronzeSchema = "01_bronze"
silverSchema = "02_silver"

bronzeTables = {
    "PRECOS_REVENDA":   f"`{CATALOGO}`.`{bronzeSchema}`.`precos_revenda`",
    "VENDAS_MUNICIPIO": f"`{CATALOGO}`.`{bronzeSchema}`.`vendas_municipio`",
    "MUNICIPIOS_IBGE":  f"`{CATALOGO}`.`{bronzeSchema}`.`municipios_ibge`",
}

silverTables = {
    "PRECOS_REVENDA": f"`{CATALOGO}`.`{silverSchema}`.`precos_revenda`",
    "VENDAS_MUNICIPIO": f"`{CATALOGO}`.`{silverSchema}`.`vendas_municipio`",
    "MUNICIPIOS": f"`{CATALOGO}`.`{silverSchema}`.`municipios`",
    "PRECOS_REJEITADOS": f"`{CATALOGO}`.`{silverSchema}`.`precos_revenda_rejeitados`",
}

##### Declaração de funções auxiliares

In [0]:
# Função auxiliar para carregar dados na tabela Delta
def mergeToDelta(df: DataFrame, tableName: str, condition: str):
    deltaTable = DeltaTable.forName(spark, tableName)

    (
        deltaTable.alias("target")
        .merge(
            df.alias("source"),
            condition,
        )
        .whenNotMatchedInsertAll()
        .execute()
    )

    print(f"[Merge-DeltaTable] Carga concluída em: {tableName}")

# Correspondência de caracteres sem acentuação
def normalizeText(column: Column) -> Column:
    accentedCharacters = "ÁÀÂÃÄÉÈÊËÍÌÎÏÓÒÔÕÖÚÙÛÜÇ"
    plainCharacters    = "AAAAAEEEEIIIIOOOOOUUUUC"

    return F.translate(F.upper(F.trim(column)), accentedCharacters, plainCharacters)

# Processa dados numéricos com vírgula como separador decimal ('99.999,99' -> '99999.99' )
def parseFloatNumbers(column: Column) -> Column:
    textValue = F.trim(column)

    return (
        F.when(
            F.instr(textValue, ",") > 0,
            F.regexp_replace(F.regexp_replace(textValue, r"\.", ""), ",", ".",)
        )
        .otherwise(textValue)
    )

# Compatibiliza nome de combustível
def fuelClassification(column: Column) -> Column:
    produto = normalizeText(column)

    return (
        F.when(produto.contains("GASOLINA"), F.lit("GASOLINA"))
        .when(produto.contains("ETANOL"), F.lit("ETANOL"))
        .when(produto.contains("DIESEL S10"), F.lit("DIESEL S10"))
        .when(produto.contains("DIESEL"), F.lit("DIESEL"))
        .when(produto.contains("GNV"), F.lit("GNV"))
        .otherwise(produto)
    )

##### Carregamento de dados de municipios [IBGE]
<p>Os municipios devem ser carregados primeiro pois seus códigos são necessários para carregamento dos dados de registro de preços.</p>

In [0]:
# Carregar dados da tabela bronze
municipiosBronzeDf = spark.table(bronzeTables["MUNICIPIOS_IBGE"])

# Seleciona e prepara dados, compatibilizando o schema da tabela Silver 'municipios_ibge'
municipiosDf = (
    municipiosBronzeDf
    .select(
        F.col("codigo_ibge").cast("long").alias("codigo_ibge"),
        F.trim(F.col("municipio")).alias("municipio"),
        F.col("codigo_uf").cast("long").alias("codigo_uf"),
        F.upper(F.trim(F.col("uf"))).alias("uf"),
        F.trim(F.col("nome_uf")).alias("nome_uf"),
        F.col("codigo_regiao").cast("long").alias("codigo_regiao"),
        F.upper(F.trim(F.col("sigla_regiao"))).alias("sigla_regiao"),
        F.trim(F.col("nome_regiao")).alias("nome_regiao"),
        F.col("codigo_regiao_imediata").cast("long").alias("codigo_regiao_imediata"),
        F.trim(F.col("regiao_imediata")).alias("regiao_imediata"),
        F.col("codigo_regiao_intermediaria")
         .cast("long")
         .alias("codigo_regiao_intermediaria"),
        F.trim(F.col("regiao_intermediaria")).alias("regiao_intermediaria"),
        F.col("data_hora_ingestao").alias("processado_bronze_em"),
        F.current_timestamp().alias("processado_silver_em"),
    )
    .filter(F.col("codigo_ibge").isNotNull())
    .dropDuplicates(["codigo_ibge"])
)

# Anexar dados a tabela delta
mergeToDelta(municipiosDf, silverTables["MUNICIPIOS"],
             "target.codigo_ibge = source.codigo_ibge",)

# Preview
display(municipiosDf.limit(10))

##### Vendas municipais ANP

In [0]:
# Carrega dados de venza da tabela Bronze
salesBronzeDf = spark.table(bronzeTables["VENDAS_MUNICIPIO"])

'''
Seleciona e prepara dados;
- Formata textos das colunas [regiao, produto]
- Formata campos numéricos [volume_vendido]
- Compatibiliza o schema da tabela Silver 'vendas_municipio'
'''
salesDf = (
    salesBronzeDf
    .select(
        F.col("ano_referencia").cast("int").alias("ano_referencia"),
        normalizeText(F.col("regiao")).alias("regiao"),
        F.upper(F.trim(F.col("uf"))).alias("uf"),
        normalizeText(F.col("produto")).alias("produto"),
        F.col("codigo_ibge").cast("long").alias("codigo_ibge"),
        F.trim(F.col("municipio")).alias("municipio"),
        parseFloatNumbers(F.col("volume_vendido")).cast("double").alias("volume_vendido"),
        F.col("arquivo_origem"),
        F.col("data_hora_ingestao").alias("processado_bronze_em"),
    )
)

# Condição necessária para uma venda ser considerada válida
evaluateValidSell = (
    F.col("ano_referencia").isNotNull()
    & F.col("codigo_ibge").isNotNull()
    & (F.length(F.col("produto")) > 0)
    & F.col("volume_vendido").isNotNull()
    & (F.col("volume_vendido") >= 0)
)

# Filtrar vendas dataframes com vendas válidas e inválidas
invalidSalesDf = salesDf.filter(~evaluateValidSell)
validSalesDf   = salesDf.filter(evaluateValidSell)

# Unir tabela de vendas com tabela de municipios através do código IBGE
salesDf = (
    validSalesDf.alias("v")
    .join(
        municipiosDf.alias("m"),
        F.col("v.codigo_ibge") == F.col("m.codigo_ibge"), 
        "left",
    )
    .select(
        F.col("v.ano_referencia"),
        normalizeText(F.coalesce(F.col("m.nome_regiao"), F.col("v.regiao"))).alias("regiao"),
        F.coalesce(F.col("m.uf"), F.col("v.uf")).alias("uf"),
        F.col("v.codigo_ibge"),
        F.coalesce(F.col("m.municipio"), F.col("v.municipio")).alias("municipio"),
        F.col("v.produto"),
        fuelClassification(F.col("v.produto")).alias("familia_combustivel"),
        F.col("v.volume_vendido"),
        F.col("v.arquivo_origem"),
        F.col("v.processado_bronze_em"),
        F.current_timestamp().alias("processado_silver_em"),
    )
    .dropDuplicates(["ano_referencia", "codigo_ibge", "produto"])
)

# Anexar dados a tabela delta
mergeToDelta(
    salesDf,
    silverTables["VENDAS_MUNICIPIO"],
    """
        target.ano_referencia = source.ano_referencia AND
        target.codigo_ibge = source.codigo_ibge AND
        target.produto = source.produto
    """,
)

print(f"Vendas inválidas desconsideradas: {invalidSalesDf.count():,}")
display(validSalesDf.limit(10))

##### Precos de revenda ANP
<p>Os dados da tabela de preços são convertidos para seus respectivos tipos e associados ao código IBGE através no nome do municipio (após normalizado) e estado.</p>
<p>Criação de uma tabela de "rejeitados", quando estes não atendem às regras mínimas de qualidade</p>

In [0]:
# Carrega dados de preços da tabela bronze
bronzePricesDf = spark.table(bronzeTables["PRECOS_REVENDA"])

'''
Seleciona e prepara dados;
- Formata textos das colunas [regiao, municipio, produto, unidade medida]
- Formata campos numéricos [valor_venda, valor_compra]
'''
pricesDf = (
    bronzePricesDf
    .select(
        normalizeText(F.col("regiao")).alias("regiao"),
        F.upper(F.trim(F.col("uf"))).alias("uf"),
        F.trim(F.col("municipio")).alias("municipio"),
        normalizeText(F.col("municipio")).alias("municipio_chave"),
        F.trim(F.col("razao_social")).alias("razao_social"),
        F.trim(F.col("cnpj_revenda")).alias("cnpj_revenda"),
        F.regexp_replace(F.col("cnpj_revenda"), r"\D", "").alias("cnpj_limpo"),
        normalizeText(F.col("produto")).alias("produto"),
        F.col("data_coleta").alias("data_coleta_original"),
        F.to_date(F.col("data_coleta"), "dd/MM/yyyy").alias("data_coleta_convertida"),
        F.col("valor_venda").alias("valor_venda_original"),
        parseFloatNumbers(F.col("valor_venda")).cast("decimal(10,3)").alias("valor_venda_convertido"),
        parseFloatNumbers(F.col("valor_compra")).cast("decimal(10,3)").alias("valor_compra_convertido"),
        normalizeText(F.col("unidade_medida")).alias("unidade_medida"),
        F.trim(F.col("bandeira")).alias("bandeira"),
        F.col("arquivo_origem"),
        F.col("data_hora_ingestao").alias("processado_bronze_em"),
    )
    .withColumn(
        "id_revenda",
        F.when(
            F.length(F.col("cnpj_limpo")) == 14,
            F.sha2(F.col("cnpj_limpo"), 256),
        ),
    )
    .withColumn("familia_combustivel", fuelClassification(F.col("produto")))
)

## Unir tabela de preços com tabela de municipios através do código IBGE
# Criar tabela chave municipio
munKeyDf = (
    municipiosDf
    .select(
        F.col("codigo_ibge"),
        F.col("municipio").alias("municipio_ibge"),
        F.col("uf"),
        normalizeText(F.col("municipio")).alias("municipio_chave"),
    )
)

# Efetuar a união da tabela de preços com a tabela de municipios
precosComMunicipioDf = (
    pricesDf.alias("p")
    .join(
        munKeyDf.alias("m"),
        (F.col("p.uf") == F.col("m.uf"))
        & (F.col("p.municipio_chave") == F.col("m.municipio_chave")),
        "left",
    )
    .select(
        F.col("p.*"),
        F.col("m.codigo_ibge"),
        F.col("m.municipio_ibge"),
    )
)

In [0]:
## Separa dados validos e inválidos
# Filtro para dados válidos
validPrice = (
    F.col("codigo_ibge").isNotNull()
    & F.col("id_revenda").isNotNull()
    & (F.length(F.col("produto")) > 0)
    & F.col("data_coleta_convertida").isNotNull()
    & F.col("valor_venda_convertido").isNotNull()
    & (F.col("valor_venda_convertido") > 0)
)

# 
rejectionCause = (
    F.when(F.col("codigo_ibge").isNull(), F.lit("MUNICIPIO_NAO_LOCALIZADO"))
    .when(F.col("id_revenda").isNull(), F.lit("CNPJ_INVALIDO"))
    .when(F.col("produto").isNull() | (F.length(F.col("produto")) == 0), F.lit("PRODUTO_INVALIDO"))
    .when(F.col("data_coleta_convertida").isNull(), F.lit("DATA_COLETA_INVALIDA"))
    .otherwise(F.lit("VALOR_VENDA_INVALIDO"))
)


rejectedPricesDf = (
    precosComMunicipioDf
    .filter(~validPrice)
    .select(
        F.col("regiao"),
        F.col("uf"),
        F.col("municipio"),
        F.col("cnpj_revenda"),
        F.col("produto"),
        F.col("data_coleta_original").alias("data_coleta"),
        F.col("valor_venda_original").alias("valor_venda"),
        rejectionCause.alias("motivo_rejeicao"),
        F.col("arquivo_origem"),
        F.current_timestamp().alias("rejeitado_em"),
    )
    .dropDuplicates([
        "cnpj_revenda",
        "produto",
        "data_coleta",
        "arquivo_origem",
        "motivo_rejeicao",
    ])
)

pricesDf = (
    precosComMunicipioDf
    .filter(validPrice)
    .select(
        F.col("regiao"),
        F.col("uf"),
        F.col("codigo_ibge"),
        F.col("municipio_ibge").alias("municipio"),
        F.col("id_revenda"),
        F.col("razao_social"),
        F.col("produto"),
        F.col("familia_combustivel"),
        F.col("data_coleta_convertida").alias("data_coleta"),
        F.col("valor_venda_convertido").alias("valor_venda"),
        F.col("valor_compra_convertido").alias("valor_compra"),
        F.col("unidade_medida"),
        F.col("bandeira"),
        F.col("arquivo_origem"),
        F.col("processado_bronze_em"),
        F.current_timestamp().alias("processado_silver_em"),
    )
    .dropDuplicates(["id_revenda", "produto", "data_coleta"])
)

mergeToDelta(pricesDf, silverTables["PRECOS_REVENDA"],
    """
        target.id_revenda = source.id_revenda AND
        target.produto = source.produto AND
        target.data_coleta = source.data_coleta
    """,
)

mergeToDelta(rejectedPricesDf, silverTables["PRECOS_REJEITADOS"],
    """
        target.cnpj_revenda <=> source.cnpj_revenda AND
        target.produto <=> source.produto AND
        target.data_coleta <=> source.data_coleta AND
        target.arquivo_origem <=> source.arquivo_origem AND
        target.motivo_rejeicao = source.motivo_rejeicao
    """,
)

display(pricesDf.limit(10))

In [0]:
display(rejectedPricesDf
        .groupBy("motivo_rejeicao")
        .count()
        .orderBy(F.desc("count")))

for tableName in silverTables.values():
    totalRows = spark.table(tableName).count()
    print(f"{tableName}: {totalRows:,} registros")